# 🎲 Bölüm 2: Olasılığın Aksiyomları
## MAT204 — Mühendisler İçin Olasılık ve İstatistik

**Konular:**
1. Örneklem Uzayı ve Olaylar
2. Küme Teorisi ve İşlemleri (∩, ∪, Aᶜ)
3. DeMorgan Yasaları
4. Olasılığın Aksiyomları (Kolmogorov)
5. Temel Önermeler (Tümleyen, Fark, İçerme-Dışlama)
6. Eşit Olasılıklı Örneklem Uzayları
7. Doğum Günü Problemi

---

In [ ]:
# Gerekli kütüphaneler
import random
import itertools
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle, FancyArrowPatch
from fractions import Fraction

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100
random.seed(42)
np.random.seed(42)

print("Kütüphaneler başarıyla yüklendi! ✓")

---
## 1. Örneklem Uzayı ve Olaylar

**Tanım:** Bir **örneklem uzayı** $S$, bir deneyin *tüm olası sonuçlarının kümesidir*.

**Tanım:** Bir **olay** $E$, örneklem uzayı $S$'nin herhangi bir alt kümesidir: $E \subseteq S$

- İmkânsız olay: $\emptyset$ (boş küme)
- Kesin olay: $S$ kendisi

In [ ]:
# Örneklem uzayı örnekleri — Python kümeleri ile temsil

# 1) Üç yazı-tura atma
S_ytura = set(itertools.product(['H','T'], repeat=3))
S_ytura_str = {''.join(s) for s in S_ytura}
print(f"3 yazı-tura örneklem uzayı S = {sorted(S_ytura_str)}")
print(f"  |S| = {len(S_ytura_str)}  (2³ = 8 ✓)")

# Olay: tam 2 tura (H)
E_2tura = {s for s in S_ytura_str if s.count('H') == 2}
print(f"  Olay E (tam 2 tura): E = {sorted(E_2tura)}, |E| = {len(E_2tura)}")

print()

# 2) Bir zar atma
S_zar = {1, 2, 3, 4, 5, 6}
E_cift = {x for x in S_zar if x % 2 == 0}
E_tek  = {x for x in S_zar if x % 2 != 0}
print(f"Bir zar: S = {sorted(S_zar)}")
print(f"  E (çift) = {sorted(E_cift)}")
print(f"  E (tek)  = {sorted(E_tek)}")
print(f"  E(çift) ∩ E(tek) = {E_cift & E_tek}  ← Karşılıklı dışlayıcı!")

print()

# 3) İki zar toplamı
S_iki_zar_ciftler = [(i, j) for i in range(1, 7) for j in range(1, 7)]
S_iki_zar = {i + j for i in range(1, 7) for j in range(1, 7)}  # toplam değerlerin kümesi
print(f"İki zar toplamı (tüm çiftler): |S| = {len(S_iki_zar_ciftler)} (6×6 = 36 eleman)")
print(f"  Olası toplam değerler: {sorted(S_iki_zar)}")

print()

# 4) Üç nükleotid dizisi
S_nukleotid = set(itertools.product('ACGT', repeat=3))
print(f"Üç nükleotid: |S| = {len(S_nukleotid)} = 4³  (A, C, G, T)")
print(f"  İlk 8 örnek: {sorted([''.join(s) for s in S_nukleotid])[:8]}...")

In [ ]:
# İki zar toplamının dağılımı görselleştirme
sonuclar = [(i, j, i+j) for i in range(1,7) for j in range(1,7)]
toplamlar = [t for _, _, t in sonuclar]
frekans   = {k: toplamlar.count(k) for k in range(2, 13)}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sol: ısı haritası (tüm çiftler)
ax = axes[0]
matris = np.array([[i+j for j in range(1,7)] for i in range(1,7)])
im = ax.imshow(matris, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(6)); ax.set_xticklabels(range(1,7))
ax.set_yticks(range(6)); ax.set_yticklabels(range(1,7))
ax.set_xlabel('2. Zar', fontsize=11)
ax.set_ylabel('1. Zar', fontsize=11)
ax.set_title('İki Zar Toplamı — Örneklem Uzayı', fontsize=12, fontweight='bold')
for i in range(6):
    for j in range(6):
        ax.text(j, i, matris[i,j], ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='white' if matris[i,j] >= 9 else 'black')
plt.colorbar(im, ax=ax, label='Toplam')

# Sağ: olasılık bar grafiği
ax2 = axes[1]
x = list(frekans.keys())
y = [v/36 for v in frekans.values()]
renkler = plt.cm.YlOrRd(np.array(y) / max(y))
bars = ax2.bar(x, y, color=renkler, edgecolor='black', linewidth=0.7)
ax2.axhline(1/6, color='blue', linestyle='--', alpha=0.5, label='1/6 referans')
for bar, val in zip(bars, y):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{val:.3f}', ha='center', va='bottom', fontsize=8)
ax2.set_xlabel('Toplam', fontsize=11)
ax2.set_ylabel('P(E)', fontsize=11)
ax2.set_title('İki Zar Toplamı Olasılıkları', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print(f"Toplam olasılık: {sum(y):.4f}  ← P(S) = 1 ✓")

---
## 2. Küme İşlemleri

| İşlem | Sembol | Anlamı |
|-------|--------|--------|
| Kesişim | $A \cap B$ | Hem A hem B gerçekleşir |
| Birleşim | $A \cup B$ | A veya B (en az biri) |
| Tümleyen | $A^c = S \setminus A$ | A gerçekleşmez |
| Alt küme | $A \subseteq B$ | A gerçekleşirse B de gerçekleşir |
| Ayrık | $A \cap B = \emptyset$ | A ve B karşılıklı dışlayıcı |

In [ ]:
# Küme işlemlerini Python ile uygulama
S = set(range(1, 7))  # Zar: {1,2,3,4,5,6}
A = {2, 4, 6}         # Çift sayılar
B = {1, 2, 3}         # 3'ten küçük veya eşit

print(f"S = {sorted(S)}")
print(f"A = {sorted(A)}  (çift)")
print(f"B = {sorted(B)}  (≤ 3)")
print()
print(f"A ∩ B = {sorted(A & B)}   (hem çift hem ≤3)")
print(f"A ∪ B = {sorted(A | B)}  (çift veya ≤3)")
print(f"Aᶜ    = {sorted(S - A)}    (çift değil = tek)")
print(f"Bᶜ    = {sorted(S - B)}  (3'ten büyük)")
print(f"A ∩ Bᶜ= {sorted(A - B)}    (çift VE 3'ten büyük)")
print()

# Kümeler arası ilişkiler
C = {2, 4}            # A'nın alt kümesi
print(f"C = {sorted(C)}  (A'nın alt kümesi mi?): C ⊆ A → {C.issubset(A)}")
print(f"A ve B ayrık mı? A ∩ B = ∅ → {len(A & B) == 0}")
D = {1, 3, 5}         # tek sayılar
print(f"D = {sorted(D)}  (tek). A ve D ayrık mı? → {len(A & D) == 0}  ✓ karşılıklı dışlayıcı")

In [ ]:
# Venn Diyagramları
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

def venn2(ax, title, shade_func, A_label='A', B_label='B'):
    ax.set_xlim(0, 10); ax.set_ylim(0, 7)
    ax.set_aspect('equal'); ax.axis('off')
    # S dikdörtgeni
    rect = plt.Rectangle((0.2, 0.5), 9.6, 6, fill=False,
                          edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(9.5, 6.3, 'S', fontsize=13, fontweight='bold')
    # Daireler
    cA = Circle((3.5, 3.5), 2.2, fill=False, edgecolor='#2166ac', linewidth=2)
    cB = Circle((6.5, 3.5), 2.2, fill=False, edgecolor='#d6604d', linewidth=2)
    ax.add_patch(cA); ax.add_patch(cB)
    ax.text(2.2, 3.5, A_label, fontsize=14, fontweight='bold', color='#2166ac', ha='center')
    ax.text(7.8, 3.5, B_label, fontsize=14, fontweight='bold', color='#d6604d', ha='center')
    # Gölgeleme
    shade_func(ax)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)

from matplotlib.patches import PathPatch
from matplotlib.path import Path

# Sol: A ∪ B
def shade_union(ax):
    cA = Circle((3.5, 3.5), 2.2, color='#92c5de', alpha=0.6, zorder=1)
    cB = Circle((6.5, 3.5), 2.2, color='#92c5de', alpha=0.6, zorder=1)
    ax.add_patch(cA); ax.add_patch(cB)
    ax.text(5, 1.0, 'Gölgeli bölge: A ∪ B', ha='center', fontsize=9, color='#0571b0')

# Orta: A ∩ B
def shade_intersection(ax):
    theta = np.linspace(0, 2*np.pi, 300)
    # Kesişim noktaları yaklaşık gösterim
    x1 = 3.5 + 2.2*np.cos(theta)
    y1 = 3.5 + 2.2*np.sin(theta)
    x2 = 6.5 + 2.2*np.cos(theta)
    y2 = 3.5 + 2.2*np.sin(theta)
    # A dairesi
    c1 = Circle((3.5, 3.5), 2.2, color='none', ec='#2166ac', lw=2)
    ax.add_patch(c1)
    # Sadece kesişimi dolgula
    from matplotlib.patches import Wedge
    c_shade = Circle((5, 3.5), 1.1, color='#f4a582', alpha=0.8, zorder=2)
    ax.add_patch(c_shade)
    ax.text(5, 1.0, 'Gölgeli bölge: A ∩ B', ha='center', fontsize=9, color='#ca0020')

# Sağ: Aᶜ
def shade_complement(ax):
    rect = plt.Rectangle((0.2, 0.5), 9.6, 6, color='#b2df8a', alpha=0.5, zorder=1)
    ax.add_patch(rect)
    cA = Circle((3.5, 3.5), 2.2, color='white', zorder=2)
    ax.add_patch(cA)
    ax.text(5, 1.0, 'Gölgeli bölge: Aᶜ', ha='center', fontsize=9, color='#33a02c')

venn2(axes[0], 'Birleşim: A ∪ B', shade_union)
venn2(axes[1], 'Kesişim: A ∩ B', shade_intersection)
venn2(axes[2], 'Tümleyen: Aᶜ', shade_complement)

plt.suptitle('Küme İşlemleri — Venn Diyagramları', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Küme Yasaları — Sayısal Doğrulama

In [ ]:
# Küme yasalarını büyük bir evren üzerinde sayısal olarak doğrula
evren = set(range(1, 21))  # S = {1,...,20}
A = {2,4,6,8,10,12,14,16,18,20}  # çift
B = {1,2,3,4,5,6,7,8,9,10}          # ≤10
C = {5,6,7,8,9,10,11,12,13,14,15}  # 5-15 arası

yasalar = [
    ("Değişmeli (∪)",    A | B,           B | A),
    ("Değişmeli (∩)",    A & B,           B & A),
    ("Birleşmeli (∪)",   (A | B) | C,     A | (B | C)),
    ("Birleşmeli (∩)",   (A & B) & C,     A & (B & C)),
    ("Dağılma 1",        (A | B) & C,     (A & C) | (B & C)),
    ("Dağılma 2",        (A & B) | C,     (A | C) & (B | C)),
    ("DeMorgan 1",       evren-(A | B),   (evren-A) & (evren-B)),
    ("DeMorgan 2",       evren-(A & B),   (evren-A) | (evren-B)),
]

print(f"{'Yasa':<20} {'Eşit mi?'}")
print("-" * 32)
for isim, sol, sag in yasalar:
    esit = sol == sag
    print(f"{isim:<20} {'✓  Doğrulandı' if esit else '✗  HATA'}")

---
## 3. DeMorgan Yasaları

$$(A \cup B)^c = A^c \cap B^c \qquad (A \cap B)^c = A^c \cup B^c$$

Genelleştirilmiş:
$$\left(\bigcup_{i=1}^n A_i\right)^c = \bigcap_{i=1}^n A_i^c \qquad \left(\bigcap_{i=1}^n A_i\right)^c = \bigcup_{i=1}^n A_i^c$$

**Sezgi:** "Ve"nin tümleyeni → "Veya"ya dönüşür; "Veya"nın tümleyeni → "Ve"ye dönüşür.

In [ ]:
# DeMorgan'ı somut örnekle göster
S = set(range(1, 13))  # {1,...,12}
A = {x for x in S if x % 2 == 0}   # çift: {2,4,6,8,10,12}
B = {x for x in S if x % 3 == 0}   # 3'ün katı: {3,6,9,12}

print(f"S = {{1,...,12}}")
print(f"A = {sorted(A)}  (çift)")
print(f"B = {sorted(B)}  (3'ün katı)")
print()

# DeMorgan 1: (A∪B)ᶜ = Aᶜ∩Bᶜ
sol1 = S - (A | B)
sag1 = (S - A) & (S - B)
print(f"DeMorgan 1: (A∪B)ᶜ = Aᶜ∩Bᶜ")
print(f"  A ∪ B = {sorted(A|B)}  (çift VEYA 3'ün katı)")
print(f"  (A∪B)ᶜ = {sorted(sol1)}  (ne çift ne 3'ün katı)")
print(f"  Aᶜ∩Bᶜ  = {sorted(sag1)}")
print(f"  Eşit mi? {sol1 == sag1}  ✓")
print()

# DeMorgan 2: (A∩B)ᶜ = Aᶜ∪Bᶜ
sol2 = S - (A & B)
sag2 = (S - A) | (S - B)
print(f"DeMorgan 2: (A∩B)ᶜ = Aᶜ∪Bᶜ")
print(f"  A ∩ B = {sorted(A&B)}  (hem çift hem 3'ün katı = 6'nın katı)")
print(f"  (A∩B)ᶜ = {sorted(sol2)}")
print(f"  Aᶜ∪Bᶜ  = {sorted(sag2)}")
print(f"  Eşit mi? {sol2 == sag2}  ✓")

# Günlük hayat yorumu
print()
print("Günlük dil yorumu:")
print("  DeMorgan 1: 'Çift VEYA 3'ün katı DEĞİL' = 'Çift DEĞİL VE 3'ün katı DEĞİL'")
print("  DeMorgan 2: 'Çift VE 3'ün katı DEĞİL' = 'Çift DEĞİL VEYA 3'ün katı DEĞİL'")

In [ ]:
# DeMorgan görselleştirmesi — n olay için genelleştirilmiş
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax_idx, (baslik, yasa_no) in enumerate([
    ("DeMorgan 1: $(A∪B)^c = A^c \\cap B^c$", 1),
    ("DeMorgan 2: $(A∩B)^c = A^c \\cup B^c$", 2)
]):
    ax = axes[ax_idx]
    ax.set_xlim(0, 10); ax.set_ylim(0, 7)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(baslik, fontsize=11, fontweight='bold')

    # S dikdörtgeni — arka plan
    if yasa_no == 1:
        bg = plt.Rectangle((0.2, 0.5), 9.6, 6, color='#92c5de', alpha=0.4, zorder=0)
    else:
        bg = plt.Rectangle((0.2, 0.5), 9.6, 6, color='#92c5de', alpha=0.4, zorder=0)
    ax.add_patch(bg)
    rect = plt.Rectangle((0.2, 0.5), 9.6, 6, fill=False, edgecolor='black', linewidth=2)
    ax.add_patch(rect)

    if yasa_no == 1:
        # Gölgeli = (A∪B)ᶜ: S'nin geri kalanı
        cA = Circle((3.5, 3.5), 2.2, color='white', zorder=2)
        cB = Circle((6.5, 3.5), 2.2, color='white', zorder=2)
        ax.add_patch(cA); ax.add_patch(cB)
        ca = Circle((3.5,3.5),2.2,fill=False,ec='#2166ac',lw=2,zorder=3)
        cb = Circle((6.5,3.5),2.2,fill=False,ec='#d6604d',lw=2,zorder=3)
        ax.add_patch(ca); ax.add_patch(cb)
        ax.text(5, 0.8, '(A∪B)ᶜ = Aᶜ∩Bᶜ (mavi bölge)', ha='center', fontsize=9)
    else:
        # Gölgeli = (A∩B)ᶜ: A ve B dışındaki kesişim hariç her şey
        cA = Circle((3.5, 3.5), 2.2, color='#92c5de', alpha=0.6, zorder=2)
        cB = Circle((6.5, 3.5), 2.2, color='#92c5de', alpha=0.6, zorder=2)
        ax.add_patch(cA); ax.add_patch(cB)
        # Kesişim beyaz (dışlanmış)
        c_int = Circle((5, 3.5), 1.05, color='white', zorder=3)
        ax.add_patch(c_int)
        ca = Circle((3.5,3.5),2.2,fill=False,ec='#2166ac',lw=2,zorder=4)
        cb = Circle((6.5,3.5),2.2,fill=False,ec='#d6604d',lw=2,zorder=4)
        ax.add_patch(ca); ax.add_patch(cb)
        ax.text(5, 0.8, '(A∩B)ᶜ = Aᶜ∪Bᶜ (mavi bölge)', ha='center', fontsize=9)

    ax.text(2.5, 3.5, 'A', fontsize=14, fontweight='bold', color='#2166ac', ha='center', zorder=5)
    ax.text(7.5, 3.5, 'B', fontsize=14, fontweight='bold', color='#d6604d', ha='center', zorder=5)
    ax.text(9.5, 6.3, 'S', fontsize=13, fontweight='bold')

plt.suptitle('DeMorgan Yasaları — Venn Diyagramı Gösterimi',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Olasılığın Aksiyomları (Kolmogorov, 1933)

$P$, her olay $E$'ye negatif olmayan bir gerçek sayı atayan fonksiyondur. $P$ bir **olasılık** olarak adlandırılır, eğer:

**Aksiyom 1 (Negatif olmayan):**
$$0 \leq P(E) \leq 1$$

**Aksiyom 2 (Toplam bir):**
$$P(S) = 1$$

**Aksiyom 3 (Sayılabilir toplama — ayrık olaylar için):**
$$P\!\left(\bigcup_{i=1}^{\infty} E_i\right) = \sum_{i=1}^{\infty} P(E_i), \quad E_i \cap E_j = \emptyset, \ i \neq j$$

In [ ]:
# Aksiyomları doğrulayan bir olasılık fonksiyonu sınıfı
class OlasilikFonksiyonu:
    def __init__(self, S, p_dict):
        """S: örneklem uzayı (set), p_dict: her sonucun olasılığı"""
        self.S = S
        self.p = p_dict
        self._aksiyomlari_dogrula()

    def _aksiyomlari_dogrula(self):
        # Aksiyom 1
        assert all(0 <= v <= 1 for v in self.p.values()), "Aksiyom 1 ihlali!"
        # Aksiyom 2
        assert abs(sum(self.p.values()) - 1.0) < 1e-9, "Aksiyom 2 ihlali!"
        print("Aksiyom 1 (0 ≤ P(e) ≤ 1): ✓")
        print(f"Aksiyom 2 (P(S) = {sum(self.p.values()):.6f}): ✓")

    def P(self, E):
        """Olay E'nin olasılığı"""
        return sum(self.p.get(e, 0) for e in E)

    def P_kesisim(self, A, B): return self.P(A & B)
    def P_birlesim(self, A, B): return self.P(A | B)
    def P_tumleyen(self, A):    return 1 - self.P(A)


# Örnek 1: Adil zar
print("=== Adil Zar ===")
S_zar = {1, 2, 3, 4, 5, 6}
p_adil = {i: 1/6 for i in S_zar}
zar = OlasilikFonksiyonu(S_zar, p_adil)

E_cift = {2, 4, 6}
E_bes_alti = {5, 6}
print(f"P(çift) = {zar.P(E_cift):.4f}  = {Fraction(zar.P(E_cift)).limit_denominator(10)}")
print(f"P(5 veya 6) = {zar.P(E_bes_alti):.4f}")
print(f"P(çiftᶜ) = P(tek) = {zar.P_tumleyen(E_cift):.4f}")
print()

# Örnek 2: Ağırlıklı (hileli) zar
print("=== Hileli Zar (6 iki kat daha olası) ===")
p_hileli = {1: 1/7, 2: 1/7, 3: 1/7, 4: 1/7, 5: 1/7, 6: 2/7}
zar_h = OlasilikFonksiyonu(S_zar, p_hileli)
print(f"P(6) = {zar_h.P({6}):.4f}  (adil zarın iki katı)")
print(f"P(çift) = {zar_h.P(E_cift):.4f}  (adil: 0.5)")

In [ ]:
# Aksiyom 3 — Sayılabilir toplama görselleştirmesi
# Zar atışında ayrık olaylar
fig, ax = plt.subplots(figsize=(11, 5))

olaylar = [
    ({1}, 'E₁={1}', '#e41a1c'),
    ({2}, 'E₂={2}', '#377eb8'),
    ({3}, 'E₃={3}', '#4daf4a'),
    ({4}, 'E₄={4}', '#984ea3'),
    ({5}, 'E₅={5}', '#ff7f00'),
    ({6}, 'E₆={6}', '#a65628'),
]

x_pos = np.arange(len(olaylar))
olasiliklar = [1/6] * 6
renkler = [o[2] for o in olaylar]

bars = ax.bar(x_pos, olasiliklar, color=renkler, edgecolor='black',
              linewidth=0.8, alpha=0.85, width=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels([o[1] for o in olaylar], fontsize=11)
ax.set_ylabel('P(Eᵢ)', fontsize=12)
ax.set_ylim(0, 0.8)
ax.set_title('Aksiyom 3: Ayrık Olayların Toplamı = P(S) = 1', fontsize=13, fontweight='bold')
ax.axhline(sum(olasiliklar), color='red', linestyle='--', linewidth=2,
           label=f'P(S) = Σ P(Eᵢ) = {sum(olasiliklar):.4f}')

# Kümülatif toplam oku
kumulatif = 0
for i, p in enumerate(olasiliklar):
    ax.text(i, p + 0.02, f'{p:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    kumulatif += p

ax.text(5.5, 0.75, f'Toplam: {sum(olasiliklar):.4f}', ha='right', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange'))
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## 5. Temel Olasılık Önermeleri

| Önerme | Formül | Koşul |
|--------|--------|-------|
| Tümleyen | $P(A^c) = 1 - P(A)$ | — |
| Boş küme | $P(\emptyset) = 0$ | — |
| Fark | $P(B \cap A^c) = P(B) - P(A)$ | $A \subseteq B$ |
| Monotonluk | $P(B) \geq P(A)$ | $A \subseteq B$ |
| İçerme-Dışlama (2) | $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ | — |
| İçerme-Dışlama (3) | $P(A∪B∪C) = P(A)+P(B)+P(C)-P(A∩B)-P(A∩C)-P(B∩C)+P(A∩B∩C)$ | — |

In [ ]:
# Tüm önermeleri sayısal olarak doğrula
# Adil zar üzerinde
P = zar.P  # kısayol
A = {2, 4, 6}    # çift
B = {4, 5, 6}    # > 3
Ac = S_zar - A

print("=" * 55)
print("Olasılık Önermelerinin Doğrulanması (Adil Zar)")
print("=" * 55)

# 1. Tümleyen
p_Ac_form  = 1 - P(A)
p_Ac_direkt = P(Ac)
print(f"\n1. Tümleyen: P(Aᶜ) = 1 - P(A)")
print(f"   P(A) = P(çift) = {P(A):.4f}")
print(f"   1-P(A) = {p_Ac_form:.4f},  P(Aᶜ) = {p_Ac_direkt:.4f}  → {'✓' if abs(p_Ac_form - p_Ac_direkt) < 1e-9 else '✗'}")

# 2. Boş küme
print(f"\n2. P(∅) = {P(set()):.4f}  → {'✓' if P(set()) == 0 else '✗'}")

# 3. Fark kuralı (A={2,4} ⊂ B={2,4,6}=A )
A2 = {2, 4};  B2 = {2, 4, 6}  # A2 ⊆ B2
print(f"\n3. Fark: P(B∩Aᶜ) = P(B)-P(A), A⊆B")
print(f"   A={sorted(A2)}, B={sorted(B2)} → A⊆B: {A2.issubset(B2)}")
sol = P(B2 & (S_zar - A2))
sag = P(B2) - P(A2)
print(f"   P(B∩Aᶜ) = {sol:.4f},  P(B)-P(A) = {sag:.4f}  → {'✓' if abs(sol-sag)<1e-9 else '✗'}")

# 4. Monotonluk
print(f"\n4. Monotonluk: P(B)≥P(A) eğer A⊆B")
print(f"   P(A={sorted(A2)}) = {P(A2):.4f} ≤ P(B={sorted(B2)}) = {P(B2):.4f}  → {'✓' if P(B2)>=P(A2) else '✗'}")

# 5. İçerme-Dışlama (2 olay)
print(f"\n5. İçerme-Dışlama (2 olay): P(A∪B) = P(A)+P(B)-P(A∩B)")
direkt = P(A | B)
formul = P(A) + P(B) - P(A & B)
print(f"   A={sorted(A)}, B={sorted(B)}")
print(f"   P(A∪B) doğrudan = {direkt:.4f}")
print(f"   P(A)+P(B)-P(A∩B) = {P(A):.4f}+{P(B):.4f}-{P(A&B):.4f} = {formul:.4f}  → {'✓' if abs(direkt-formul)<1e-9 else '✗'}")

In [ ]:
# Slayttaki örnek: Olasılık sınıfı öğrencisi
print("=" * 55)
print("İçerme-Dışlama Örneği — Olasılık Sınıfı")
print("=" * 55)

p_A       = 0.63   # P(Doğu zaman dilimi)
p_B       = 0.41   # P(Son sınıf)
p_AB      = 0.31   # P(Her ikisi)

p_AuB = p_A + p_B - p_AB

print(f"P(A) = P(Doğu zaman dilimi) = {p_A}")
print(f"P(B) = P(Son sınıf)         = {p_B}")
print(f"P(A∩B) = P(Her ikisi)       = {p_AB}")
print(f"\nP(A∪B) = {p_A} + {p_B} - {p_AB} = {p_AuB}")

# Kontroller
p_Ac      = 1 - p_A
p_Bc      = 1 - p_B
p_Ac_Bc   = 1 - p_AuB   # DeMorgan
p_sadece_A= p_A - p_AB
p_sadece_B= p_B - p_AB

print(f"\nEk hesaplamalar:")
print(f"  P(Aᶜ) = {p_Ac:.2f}  (Doğu zaman diliminde değil)")
print(f"  P(Bᶜ) = {p_Bc:.2f}  (Son sınıf değil)")
print(f"  P(Ne A ne B) = 1-P(A∪B) = {p_Ac_Bc:.2f}")
print(f"  P(Sadece A) = {p_sadece_A:.2f}")
print(f"  P(Sadece B) = {p_sadece_B:.2f}")
print(f"  Toplam: {p_sadece_A:.2f}+{p_sadece_B:.2f}+{p_AB:.2f}+{p_Ac_Bc:.2f} = {p_sadece_A+p_sadece_B+p_AB+p_Ac_Bc:.2f}  ✓")

In [ ]:
# İçerme-Dışlama görselleştirmesi
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sol: Öğrenci örneği Venn diyagramı
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 7); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Öğrenci Örneği — İçerme-Dışlama', fontsize=11, fontweight='bold')

rect = plt.Rectangle((0.2, 0.5), 9.6, 6, fill=False, edgecolor='black', linewidth=2)
ax.add_patch(rect)
cA = Circle((3.5, 3.5), 2.4, color='#92c5de', alpha=0.6, zorder=2)
cB = Circle((6.5, 3.5), 2.4, color='#f4a582', alpha=0.6, zorder=2)
ax.add_patch(cA); ax.add_patch(cB)
ca = Circle((3.5,3.5),2.4,fill=False,ec='#2166ac',lw=2,zorder=3)
cb = Circle((6.5,3.5),2.4,fill=False,ec='#ca0020',lw=2,zorder=3)
ax.add_patch(ca); ax.add_patch(cb)

ax.text(2.3, 3.5, f'A sadece\n{p_sadece_A:.2f}', ha='center', fontsize=9, fontweight='bold', zorder=4)
ax.text(5.0, 3.5, f'A∩B\n{p_AB:.2f}', ha='center', fontsize=9, fontweight='bold', zorder=4)
ax.text(7.7, 3.5, f'B sadece\n{p_sadece_B:.2f}', ha='center', fontsize=9, fontweight='bold', zorder=4)
ax.text(1.2, 1.2, f'Hiçbiri\n{p_Ac_Bc:.2f}', ha='center', fontsize=9, zorder=4)
ax.text(2.0, 5.8, 'A=Doğu\nZaman', ha='center', fontsize=8, color='#2166ac')
ax.text(8.0, 5.8, 'B=Son\nSınıf', ha='center', fontsize=8, color='#ca0020')
ax.text(9.5, 6.3, 'S', fontsize=13, fontweight='bold')
ax.text(5, 0.2, f'P(A∪B) = {p_AuB:.2f}', ha='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow'))

# Sağ: 3 olay İçerme-Dışlama
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, 7.5); ax2.set_aspect('equal'); ax2.axis('off')
ax2.set_title('3 Olay — İçerme-Dışlama\nP(A∪B∪C)', fontsize=11, fontweight='bold')

rect2 = plt.Rectangle((0.2, 0.5), 9.6, 6.5, fill=False, edgecolor='black', linewidth=2)
ax2.add_patch(rect2)

cA2 = Circle((4.0, 4.2), 2.0, color='#92c5de', alpha=0.45, zorder=2)
cB2 = Circle((6.0, 4.2), 2.0, color='#f4a582', alpha=0.45, zorder=2)
cC2 = Circle((5.0, 2.4), 2.0, color='#b2df8a', alpha=0.45, zorder=2)
for c in [cA2, cB2, cC2]: ax2.add_patch(c)
for (cx,cy,r,col) in [(4.0,4.2,2.0,'#2166ac'),(6.0,4.2,2.0,'#ca0020'),(5.0,2.4,2.0,'#33a02c')]:
    ax2.add_patch(Circle((cx,cy),r,fill=False,ec=col,lw=2,zorder=3))

ax2.text(3.0, 5.2, 'A', fontsize=13, fontweight='bold', color='#2166ac', ha='center')
ax2.text(7.0, 5.2, 'B', fontsize=13, fontweight='bold', color='#ca0020', ha='center')
ax2.text(5.0, 1.2, 'C', fontsize=13, fontweight='bold', color='#33a02c', ha='center')
ax2.text(5, 3.4, 'A∩B∩C', ha='center', fontsize=7, fontweight='bold', zorder=4)
ax2.text(9.5, 6.8, 'S', fontsize=13, fontweight='bold')
ax2.text(5, 0.2,
         'P(A∪B∪C) = P(A)+P(B)+P(C)\n  −P(A∩B)−P(A∩C)−P(B∩C)+P(A∩B∩C)',
         ha='center', fontsize=8,
         bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.tight_layout()
plt.show()

---
## 6. Eşit Olasılıklı Örneklem Uzayları

Örneklem uzayı $N$ eşit olasılıklı sonuç içeriyorsa:

$$P(\{i\}) = \frac{1}{N} \qquad \Rightarrow \qquad P(E) = \frac{\#(E)}{\#(S)}$$

In [ ]:
def esit_olasilik(S, E):
    """Eşit olasılıklı uzayda P(E) = |E|/|S|"""
    return len(E) / len(S)

print("=" * 50)
print("Eşit Olasılıklı Uzay Örnekleri")
print("=" * 50)

# 1. Zar: çift sayı
S1 = {1,2,3,4,5,6}
E1 = {2,4,6}
p1 = esit_olasilik(S1, E1)
print(f"\n1. Zar: Çift sayı")
print(f"   S={sorted(S1)}, E={sorted(E1)}")
print(f"   P(E) = {len(E1)}/{len(S1)} = {p1:.4f} = {Fraction(len(E1), len(S1))}")

# 2. İki çocuk: ikisi de kız değil
S2 = {'BB', 'BG', 'GB', 'GG'}
E2 = {'BB', 'BG', 'GB'}  # en az biri erkek
p2 = esit_olasilik(S2, E2)
print(f"\n2. İki çocuk: en az biri erkek")
print(f"   S={sorted(S2)}, E={sorted(E2)}")
print(f"   P(E) = {len(E2)}/{len(S2)} = {p2:.4f} = {Fraction(len(E2), len(S2))}")

# 3. Yazı-tura: tam 2 tura
S3 = {''.join(s) for s in itertools.product('HT', repeat=3)}
E3 = {s for s in S3 if s.count('H') == 2}
p3 = esit_olasilik(S3, E3)
print(f"\n3. 3 yazı-tura: tam 2 tura")
print(f"   |S| = {len(S3)}, E = {sorted(E3)}")
print(f"   P(E) = {len(E3)}/{len(S3)} = {p3:.4f} = {Fraction(len(E3), len(S3))}")

# 4. Poker: en az 1 as
toplam_el = math.comb(52, 5)
en_az_1_as = toplam_el - math.comb(48, 5)  # en az 1 as = toplam - (hiç as yok)
p4 = en_az_1_as / toplam_el
print(f"\n4. Poker (52 kart, 5 çek): en az 1 as")
print(f"   C(52,5) = {toplam_el:,}")
print(f"   P(en az 1 as) = 1 - P(hiç as yok) = 1 - C(48,5)/C(52,5)")
print(f"                = {p4:.5f} ≈ {p4*100:.2f}%")

---
## 7. Doğum Günü Problemi 🎂

$n$ kişi arasında hiç doğum günü çakışmaması olasılığı:

$$P(\text{eşleşme yok}) = \frac{365!}{(365-n)! \cdot 365^n} = \frac{365}{365} \cdot \frac{364}{365} \cdots \frac{365-n+1}{365}$$

$$P(\text{en az bir eşleşme}) = 1 - P(\text{eşleşme yok})$$

**Sürpriz:** $n = 23$ için bile $P(\text{eşleşme}) > 0.5$!

In [ ]:
def p_eslesme_yok(n, gun=365):
    """n kişi arasında hiç doğum günü eşleşmemesi olasılığı"""
    if n > gun:
        return 0.0
    p = 1.0
    for k in range(n):
        p *= (gun - k) / gun
    return p

def p_eslesme_var(n, gun=365):
    return 1 - p_eslesme_yok(n, gun)

# Tablo
print(f"{'n':>4} | {'P(eşleşme yok)':>18} | {'P(en az 1 eşleşme)':>20}")
print("-" * 48)
ilginc_n = [5, 10, 15, 20, 23, 25, 30, 40, 50, 60, 70]
for n in ilginc_n:
    p_yok = p_eslesme_yok(n)
    p_var = p_eslesme_var(n)
    isaretler = " ← %50 eşik!" if n == 23 else ""
    print(f"{n:>4} | {p_yok:>18.6f} | {p_var:>18.6f}{isaretler}")

In [ ]:
# Doğum günü problemi — görselleştirme
n_deger = np.arange(1, 80)
p_yok_arr = [p_eslesme_yok(n) for n in n_deger]
p_var_arr = [p_eslesme_var(n) for n in n_deger]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: olasılık eğrileri
ax = axes[0]
ax.plot(n_deger, p_yok_arr, 'b-', linewidth=2.5, label='P(eşleşme yok)')
ax.plot(n_deger, p_var_arr, 'r-', linewidth=2.5, label='P(en az 1 eşleşme)')
ax.axhline(0.5, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='P = 0.5')
ax.axvline(23, color='purple', linestyle=':', linewidth=2, alpha=0.8, label='n = 23')

# 23. noktayı işaretle
ax.scatter([23], [p_eslesme_var(23)], color='red', s=120, zorder=5)
ax.annotate(f'n=23\nP={p_eslesme_var(23):.3f}',
            xy=(23, p_eslesme_var(23)),
            xytext=(35, 0.45),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow'))

ax.set_xlabel('Kişi sayısı (n)', fontsize=12)
ax.set_ylabel('Olasılık', fontsize=12)
ax.set_title('Doğum Günü Problemi', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

# Sağ: Monte Carlo simülasyonu ile doğrulama
ax2 = axes[1]

def monte_carlo_dogum(n, deneysayisi=10000):
    eslesme_sayisi = 0
    for _ in range(deneysayisi):
        dogumgunler = np.random.randint(1, 366, size=n)
        if len(dogumgunler) != len(set(dogumgunler)):
            eslesme_sayisi += 1
    return eslesme_sayisi / deneysayisi

np.random.seed(42)
n_test = [5, 10, 15, 20, 23, 30, 40, 50]
mc_sonuc    = [monte_carlo_dogum(n, 8000) for n in n_test]
teori_sonuc = [p_eslesme_var(n) for n in n_test]

x_pos = np.arange(len(n_test))
genislik = 0.35
ax2.bar(x_pos - genislik/2, teori_sonuc, genislik, label='Teorik',
        color='#4C72B0', alpha=0.85, edgecolor='black', linewidth=0.5)
ax2.bar(x_pos + genislik/2, mc_sonuc, genislik, label='Monte Carlo (8000 deneme)',
        color='#DD8452', alpha=0.85, edgecolor='black', linewidth=0.5)
ax2.axhline(0.5, color='green', linestyle='--', linewidth=1.5, alpha=0.7)
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'n={n}' for n in n_test], rotation=30, ha='right')
ax2.set_ylabel('P(en az 1 eşleşme)', fontsize=11)
ax2.set_title('Teorik vs Monte Carlo Doğrulaması', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

print("\nKarşılaştırma:")
print(f"{'n':>4} | {'Teorik':>10} | {'Monte Carlo':>12} | {'Fark':>8}")
print("-" * 44)
for n, t, mc in zip(n_test, teori_sonuc, mc_sonuc):
    print(f"{n:>4} | {t:>10.5f} | {mc:>12.5f} | {abs(t-mc):>8.5f}")

---
## 8. Olasılık Yorumları

**Sıklıkçı:** $P(A) = \lim_{n \to \infty} \dfrac{\text{A'nın gerçekleşme sayısı}}{n}$

**Bayesçi:** P(A), kişisel inanç derecesidir; yeni verilerle **Bayes Kuralı** ile güncellenir.

In [ ]:
# Sıklıkçı yorumlama — yazı-tura simülasyonu
np.random.seed(42)
N_max = 10_000
atislar = np.random.randint(0, 2, size=N_max)  # 0=Yazı, 1=Tura

# Kümülatif tura oranı
n_dizi = np.arange(1, N_max + 1)
kumulatif_tura = np.cumsum(atislar == 1) / n_dizi

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: yakınsama
ax = axes[0]
ax.semilogx(n_dizi, kumulatif_tura, color='#2166ac', linewidth=1.5, alpha=0.9,
            label='Gözlemlenen oran')
ax.axhline(0.5, color='red', linestyle='--', linewidth=2, label='Teorik P(Tura) = 0.5')
ax.fill_between(n_dizi,
                0.5 - 1/np.sqrt(n_dizi),
                0.5 + 1/np.sqrt(n_dizi),
                alpha=0.15, color='red', label='±1/√n güven bandı')
ax.set_xlabel('Atış sayısı (log ölçeği)', fontsize=11)
ax.set_ylabel('Kümülatif Tura Oranı', fontsize=11)
ax.set_title('Sıklıkçı Yorum — Büyük Sayılar Kanunu', fontsize=12, fontweight='bold')
ax.set_ylim(0.3, 0.7)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Sağ: Birkaç farklı başlangıç
ax2 = axes[1]
np.random.seed(0)
for deney_no in range(8):
    atislar_i = np.random.randint(0, 2, size=N_max)
    kum_i     = np.cumsum(atislar_i == 1) / n_dizi
    ax2.semilogx(n_dizi, kum_i, alpha=0.5, linewidth=1)
ax2.axhline(0.5, color='black', linestyle='--', linewidth=2.5, label='P = 0.5')
ax2.set_xlabel('Atış sayısı (log ölçeği)', fontsize=11)
ax2.set_ylabel('Kümülatif Tura Oranı', fontsize=11)
ax2.set_title('8 Bağımsız Simülasyon — Hepsi P=0.5\'e Yakınsıyor', fontsize=12, fontweight='bold')
ax2.set_ylim(0.3, 0.7)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Son değer (10000 atış): P̂(Tura) = {kumulatif_tura[-1]:.5f}  (teorik: 0.5)")

---
## 9. Özet

In [ ]:
print("=" * 65)
print("BÖLÜM 2 ÖZET — Olasılığın Aksiyomları")
print("=" * 65)

ozet = [
    ("Örneklem Uzayı S",   "Tüm olası sonuçların kümesi"),
    ("Olay E",             "E ⊆ S (örneklem uzayının alt kümesi)"),
    ("Kesişim A∩B",        "Hem A hem B"),
    ("Birleşim A∪B",       "A veya B (en az biri)"),
    ("Tümleyen Aᶜ",        "A gerçekleşmez = S\\A"),
    ("Ayrık olaylar",      "A∩B = ∅"),
    ("DeMorgan 1",         "(A∪B)ᶜ = Aᶜ∩Bᶜ"),
    ("DeMorgan 2",         "(A∩B)ᶜ = Aᶜ∪Bᶜ"),
    ("Aksiyom 1",          "0 ≤ P(E) ≤ 1"),
    ("Aksiyom 2",          "P(S) = 1"),
    ("Aksiyom 3",          "P(∪Eᵢ) = ΣP(Eᵢ)  (ayrık olaylar)"),
    ("Tümleyen kuralı",    "P(Aᶜ) = 1 − P(A)"),
    ("İçerme-Dışlama",     "P(A∪B) = P(A)+P(B)−P(A∩B)"),
    ("Eşit olasılıklı",    "P(E) = |E| / |S|"),
    ("Doğum günü",         "n=23'te P(eşleşme)>0.5  (sürpriz!)"),
]

print(f"{'Kavram':<22} | {'Açıklama / Formül'}")
print("-" * 65)
for kavram, aciklama in ozet:
    print(f"{kavram:<22} | {aciklama}")
print("=" * 65)

---
## 10. Alıştırmalar

**Soru 1:** Bir zar atışında $P(\text{5'ten büyük veya çift})$ nedir?

**Soru 2:** Bir sınıfta 30 öğrenci var. Hiç doğum günü eşleşmemesi olasılığı nedir?

**Soru 3:** $P(A) = 0.4$, $P(B) = 0.5$, $P(A \cap B) = 0.2$ ise $P(A^c \cup B^c)$ nedir?

**Soru 4:** 52 kartlı desteden 5 kart çekildiğinde, en az bir kupa olasılığı nedir?

In [ ]:
# CEVAPLAR
print("SORU 1: P(5'ten büyük VEYA çift)")
S1 = {1,2,3,4,5,6}
A1 = {6}        # 5'ten büyük
B1 = {2,4,6}    # çift
p_s1 = (len(A1) + len(B1) - len(A1&B1)) / len(S1)
print(f"  P(A∪B) = P({sorted(A1)}∪{sorted(B1)}) = {p_s1:.4f} = {Fraction(len(A1)+len(B1)-len(A1&B1), len(S1))}")

print()
print("SORU 2: 30 öğrencide hiç doğum günü eşleşmemesi")
p_s2 = p_eslesme_yok(30)
print(f"  P(eşleşme yok, n=30) = {p_s2:.6f} ≈ {p_s2*100:.2f}%")
print(f"  P(en az 1 eşleşme)   = {1-p_s2:.6f} ≈ {(1-p_s2)*100:.2f}%")

print()
print("SORU 3: P(Aᶜ∪Bᶜ) = P((A∩B)ᶜ)  [DeMorgan]")
pA, pB, pAB = 0.4, 0.5, 0.2
p_s3 = 1 - pAB  # DeMorgan: (A∩B)ᶜ
print(f"  P(Aᶜ∪Bᶜ) = P((A∩B)ᶜ) = 1 - P(A∩B) = 1 - {pAB} = {p_s3}")

print()
print("SORU 4: 52 karttan 5 çekildiğinde en az 1 kupa")
toplam = math.comb(52, 5)
hic_kupa_yok = math.comb(39, 5)  # 52-13=39 kart kupa değil
p_s4 = 1 - hic_kupa_yok / toplam
print(f"  P(en az 1 kupa) = 1 - C(39,5)/C(52,5)")
print(f"                  = 1 - {hic_kupa_yok:,}/{toplam:,}")
print(f"                  = {p_s4:.5f} ≈ {p_s4*100:.2f}%")